# SORT Tracker

## Imports

In [3]:
# --- imports
import warnings
from pathlib import Path

import cv2
import numpy as np
import supervision as sv
from trackers import SORTTracker
from trackers.utils.iou import BIoU, CIoU, GIoU, IoU

from orc_model.data import ClipDataset
from orc_model.components.detector import Detector

# unrelated internal deprecation noise from `trackers` itself (a `target=None`
# proxy warning raised on import), not anything about our usage
warnings.filterwarnings("ignore", category=FutureWarning, module="trackers")


# --- configs
# detector
WEIGHTS_PATH = Path("../../weights/rf-detr-seg-2xl-v1.onnx")
CONFIDENCE_THRESHOLD = 0.80

# data
CLIP_NAME = "IMG_2081"
TARGET_FPS = 30

# outputs
IOU_CLASS = GIoU #
IOU_TYPE = IOU_CLASS.__name__
OUTPUT_VIDEO_PATH = Path("../../artifacts") / f"{CLIP_NAME}_sort_tracked_{TARGET_FPS}fps_{IOU_TYPE}.mp4"

## Helpers

In [4]:
# --- resample 
def sample_frame_indices(native_fps: float, frame_count: int, target_fps: float) -> list[int]:
    """Evenly-spaced frame indices approximating `target_fps` given the video's native fps."""
    step = max(round(native_fps / target_fps), 1)
    return list(range(0, frame_count, step))

def read_frames(video_path: Path, frame_indices: list[int]) -> list[np.ndarray]:
    """Sequential decode + pick, rather than repeated `cap.set(...)` seeks."""
    wanted = set(frame_indices)
    cap = cv2.VideoCapture(str(video_path))
    frames_by_index = {}
    index = 0
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        if index in wanted:
            frames_by_index[index] = frame
        index += 1
    cap.release()
    return [frames_by_index[i] for i in frame_indices if i in frames_by_index]

## Data

In [5]:
# --- get the data
dataset = ClipDataset.from_data_dir()
clip = dataset.get_clip(CLIP_NAME)

print(f"video: {clip.video_path}")
print(f"native fps: {clip.fps}, frame_count: {clip.frame_count}, resolution: {clip.resolution}")

# --- resample
frame_indices = sample_frame_indices(clip.fps, clip.frame_count, TARGET_FPS)
frames = read_frames(clip.video_path, frame_indices)

video: /Users/constantijn/Documents/eTHi.Link/dev/MVP-OKcamera/model/data/IMG_2081/video/IMG_2081.mp4
native fps: 30.0, frame_count: 1859, resolution: (1920, 1080)


## Detect + track

Run RF-DETR on each sampled frame, feed its `sv.Detections` into `SORTTracker.update` in temporal order.

In [6]:
# --- models
detector = Detector(WEIGHTS_PATH, confidence_threshold=CONFIDENCE_THRESHOLD)

# --- tracker
from trackers.utils.state_representations import XCYCSRStateEstimator, XYXYStateEstimator
tracker = SORTTracker(
    frame_rate=TARGET_FPS,
    track_activation_threshold=0.90,
    minimum_consecutive_frames=2*TARGET_FPS,
    lost_track_buffer=20*TARGET_FPS,
    minimum_iou_threshold=-0.30,
    state_estimator_class=XCYCSRStateEstimator,
    iou=IOU_CLASS(),
)

# --- annots
mask_annotator = sv.MaskAnnotator(opacity=0.5, color_lookup=sv.ColorLookup.TRACK)
box_annotator = sv.BoxAnnotator(color_lookup=sv.ColorLookup.TRACK)
label_annotator = sv.LabelAnnotator(color_lookup=sv.ColorLookup.TRACK)


In [ ]:
annotated_frames = []
track_ids_per_frame = []  # per-frame set of active track ids, for the summary below
track_positions_per_frame = []  # per-frame dict of tracker_id -> (x_center, y_center)

for image in frames:
    detections = detector.predict(image)
    tracked = tracker.update(detections)

    confidences = getattr(tracked, "confidence", None)
    if confidences is not None:
        labels = [
            f"#{tracker_id} {float(confidence):.2f}"
            for tracker_id, confidence in zip(tracked.tracker_id, confidences, strict=False)
        ]
    else:
        labels = [f"#{tracker_id}" for tracker_id in tracked.tracker_id]
    annotated = image.copy()
    annotated = mask_annotator.annotate(annotated, tracked)
    annotated = box_annotator.annotate(annotated, tracked)
    annotated = label_annotator.annotate(annotated, tracked, labels=labels)

    annotated_frames.append(annotated)
    track_ids_per_frame.append(set(tracked.tracker_id.tolist()))

    positions = {}
    for tracker_id, xyxy in zip(tracked.tracker_id, tracked.xyxy, strict=False):
        x_center = float(xyxy[0] + (xyxy[2] - xyxy[0]) / 2)
        y_center = float(xyxy[1] + (xyxy[3] - xyxy[1]) / 2)
        positions[int(tracker_id)] = (x_center, y_center)

    track_positions_per_frame.append(positions)

print(f"tracked {len(annotated_frames)} frames")

## Track persistence

In [ ]:
all_track_ids = sorted(set().union(*track_ids_per_frame)) if track_ids_per_frame else []
print(f"{len(all_track_ids)} unique track ids observed over {len(frames)} sampled frames\n")

# for track_id in all_track_ids:
#     frame_hits = sum(track_id in ids for ids in track_ids_per_frame)
#     print(f"  track #{track_id}: present in {frame_hits}/{len(frames)} sampled frames")

# Show a compact timeline of a small subset of tracks over time.
import plotly.graph_objects as go

track_ids_to_plot = all_track_ids#[: min(6, len(all_track_ids))]
frame_positions = np.array(frame_indices)

fig = go.Figure()
for track_id in track_ids_to_plot:
    hits = [idx for idx, ids in enumerate(track_ids_per_frame) if track_id in ids]
    if hits:
        fig.add_trace(
            go.Scatter(
                x=[frame_positions[idx] for idx in hits],
                y=[track_id] * len(hits),
                mode="markers",
                marker=dict(size=8),
                name=f"track #{track_id}",
            )
        )

fig.update_layout(
    title="Track presence over sampled frames",
    xaxis_title="frame index",
    yaxis_title="track id",
    height=320,
    margin=dict(l=40, r=20, t=40, b=40),
)
fig.show()

15 unique track ids observed over 930 sampled frames



In [ ]:
# Also show the trajectory of a specific track in image coordinates.
# Pick the first track that appears at least twice and use its frame-by-frame positions.
track_id_for_trajectory = 11

if track_id_for_trajectory is not None:
    trajectory = []
    for frame_idx, positions in enumerate(track_positions_per_frame):
        if track_id_for_trajectory in positions:
            x_center, y_center = positions[track_id_for_trajectory]
            trajectory.append((frame_positions[frame_idx], x_center, y_center))

    if trajectory:
        trajectory_arr = np.array(trajectory)
        fig2 = go.Figure()
        fig2.add_trace(
            go.Scatter(
                x=trajectory_arr[:, 1],
                y=trajectory_arr[:, 2],
                mode="lines+markers",
                marker=dict(size=8),
                line=dict(width=2),
                name=f"trajectory of track #{track_id_for_trajectory}",
            )
        )
        fig2.update_layout(
            title=f"Trajectory of track #{track_id_for_trajectory}",
            xaxis_title="x (pixels)",
            yaxis_title="y (pixels)",
            height=600,
            width=600,
            margin=dict(l=40, r=20, t=40, b=40),
        )
        fig2.show()
    else:
        print("No trajectory points were found for the selected track.")
else:
    print("No track with at least two appearances was found.")

In [ ]:
OUTPUT_VIDEO_PATH.parent.mkdir(parents=True, exist_ok=True)

video_info = sv.VideoInfo(
    width=clip.resolution[0],
    height=clip.resolution[1],
    fps=TARGET_FPS,
    total_frames=len(annotated_frames),
)

with sv.VideoSink(str(OUTPUT_VIDEO_PATH), video_info) as sink:
    for annotated in annotated_frames:
        sink.write_frame(annotated)

print(f"wrote {OUTPUT_VIDEO_PATH}")

wrote ../../artifacts/IMG_2081_sort_tracked_15fps_GIoU.mp4
